# FGES 在 ER 图上的 SHD 扫描：不同 d 与 degree

固定噪声类型与样本量，扫描多组 (d, degree)，在每组配置下重复若干次 ER 图实验，运行 FGES 并记录 SHD。

- **FGES**：Fast Greedy Equivalence Search，via `pytetrad` (`fges_compat`)，需 JDK 21+

最终将 SHD 平均值以 (degree × d) 的透视表形式输出。

In [8]:
# 1) 环境与导入
import os
import sys
import time
import logging
from datetime import datetime

import numpy as np
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
if not os.environ.get('JAVA_HOME'):
    _jdk_candidate = r'C:\Program Files\Microsoft\jdk-21.0.10.7-hotspot'
    if os.path.isdir(_jdk_candidate):
        os.environ['JAVA_HOME'] = _jdk_candidate

print('Python   :', sys.version.split()[0])
print('Repo root:', repo_root)

from MEC import is_in_markov_equiv_class, get_skeleton, find_v_structures
from synthetic_dataset import SyntheticDataset

# ── FGES (pytetrad, requires JDK 21+) ────────────────────────────────────────
FGES_IMPORT_ERROR = None
try:
    import fges_compat as _ts
    _probe = pd.DataFrame(np.eye(2), columns=['x0', 'x1'])
    _s = _ts.TetradSearch(_probe)
    _s.set_verbose(False)
    _s.use_sem_bic()
    _s.run_fges()
    del _probe, _s
    HAS_FGES = True
    print('FGES      : OK')
except Exception as _e:
    HAS_FGES = False
    FGES_IMPORT_ERROR = _e
    print('FGES unavailable:', _e)

Python   : 3.10.11
Repo root: c:\Users\super\DAG
FGES      : OK


In [9]:
# 2) 辅助函数

def weight_to_binary_adj(W: np.ndarray, threshold: float = 0.0) -> np.ndarray:
    G = (np.abs(W) > threshold).astype(int)
    np.fill_diagonal(G, 0)
    return G


def shd_score(G_true: np.ndarray, G_est: np.ndarray) -> float:
    G_true = np.asarray(G_true, dtype=int)
    G_est  = np.asarray(G_est,  dtype=int)
    d, dist = G_true.shape[0], 0
    for i in range(d):
        for j in range(i + 1, d):
            if G_true[i, j] != G_est[i, j] or G_true[j, i] != G_est[j, i]:
                dist += 1
    return float(dist)


def fges_matrix_to_adj(df_result) -> np.ndarray:
    """pytetrad get_graph_to_matrix() -> 0/1 adjacency (ARROW=2, TAIL=3)."""
    mat = df_result.values
    d   = mat.shape[0]
    G   = np.zeros((d, d), dtype=int)
    for i in range(d):
        for j in range(i + 1, d):
            a, b = int(mat[i, j]), int(mat[j, i])
            if   a == 2 and b == 3: G[i, j] = 1
            elif a == 3 and b == 2: G[j, i] = 1
            elif a == 3 and b == 3: G[i, j] = G[j, i] = 1
            elif a != 0 or  b != 0: G[i, j] = G[j, i] = 1
    np.fill_diagonal(G, 0)
    return G


print('辅助函数已定义。')

辅助函数已定义。


In [10]:
# 3) 实验配置
CFG = {
    'trials':       20,
    'seed':         42,

    # 扫描网格：d × degree
    'd_list':       [20, 30, 40, 50],
    'degree_list':  [1.0, 2.0, 3.0, 4.0],

    # 固定项
    'n':            50000,
    'noise_type':   'gaussian_ev',
    'b_scale':      1.0,

    # FGES
    'fges_penalty_discount': 1.0,

    # 输出
    'out_dir': os.path.join(repo_root, 'experiments', 'results'),
    'tag':     'er_fges_d_degree',
}
os.makedirs(CFG['out_dir'], exist_ok=True)

print('配置完成。')
print(f"  d_list      : {CFG['d_list']}")
print(f"  degree_list : {CFG['degree_list']}")
print(f"  n           : {CFG['n']}")
print(f"  noise_type  : {CFG['noise_type']}")
print(f"  trials      : {CFG['trials']}")
print(f"  网格规模    : {len(CFG['d_list'])} × {len(CFG['degree_list'])} × {CFG['trials']}"
      f" = {len(CFG['d_list']) * len(CFG['degree_list']) * CFG['trials']} trials")

配置完成。
  d_list      : [20, 30, 40, 50]
  degree_list : [1.0, 2.0, 3.0, 4.0]
  n           : 50000
  noise_type  : gaussian_ev
  trials      : 20
  网格规模    : 4 × 4 × 20 = 320 trials


In [11]:
# 4) 运行 FGES 扫描
# ──────────────────────────────────────────────────────────────────────────────

_rows = []
_SKIP_LOGS = []

if not HAS_FGES:
    print(f'FGES 不可用，跳过运行。错误：{FGES_IMPORT_ERROR}')
else:
    rng = np.random.default_rng(CFG['seed'])
    for _d in CFG['d_list']:
        for _deg in CFG['degree_list']:
            _seeds = rng.integers(0, 10**9, size=CFG['trials'])
            for _tid, _seed in enumerate(_seeds, start=1):
                try:
                    _ds = SyntheticDataset(
                        n=CFG['n'], d=_d,
                        graph_type='ER',
                        degree=_deg,
                        noise_type=CFG['noise_type'],
                        B_scale=CFG['b_scale'],
                        seed=int(_seed),
                    )
                    _X      = _ds.X
                    _G_true = weight_to_binary_adj(_ds.B, threshold=0.0)

                    _cols   = [f'x{_i}' for _i in range(_d)]
                    _df_X   = pd.DataFrame(_X, columns=_cols).astype('float64')

                    _t0     = time.perf_counter()
                    _search = _ts.TetradSearch(_df_X)
                    _search.set_verbose(False)
                    _search.use_sem_bic(penalty_discount=CFG['fges_penalty_discount'])
                    _search.run_fges()
                    _rt     = time.perf_counter() - _t0
                    _G_est  = fges_matrix_to_adj(_search.get_graph_to_matrix())

                    _shd = shd_score(_G_true, _G_est)
                    _row = {
                        'd': _d,
                        'degree': _deg,
                        'trial_id': _tid,
                        'seed': int(_seed),
                        'shd': _shd,
                        'n_edges_true': int(_G_true.sum()),
                        'n_edges_est':  int(_G_est.sum()),
                        'runtime_sec':  float(_rt),
                    }
                    _rows.append(_row)
                    print(f'[fges] d={_d} deg={_deg} trial={_tid}  '
                          f'shd={_shd:.0f}  edges_true={_row["n_edges_true"]}  '
                          f'edges_est={_row["n_edges_est"]}  rt={_rt:.3f}s')
                except Exception as _e:
                    _SKIP_LOGS.append({'d': _d, 'degree': _deg, 'trial_id': _tid,
                                       'reason': str(_e)})
                    print(f'[SKIP] d={_d} deg={_deg} trial={_tid}: {_e}')

df_trials = pd.DataFrame(_rows)
print(f'\n总 trial 数: {len(df_trials)}  |  跳过数: {len(_SKIP_LOGS)}')

[fges] d=20 deg=1.0 trial=1  shd=5  edges_true=12  edges_est=17  rt=2.765s
[fges] d=20 deg=1.0 trial=2  shd=6  edges_true=13  edges_est=19  rt=2.782s
[fges] d=20 deg=1.0 trial=3  shd=1  edges_true=10  edges_est=11  rt=2.687s
[fges] d=20 deg=1.0 trial=4  shd=5  edges_true=13  edges_est=18  rt=2.926s
[fges] d=20 deg=1.0 trial=5  shd=1  edges_true=3  edges_est=4  rt=2.813s
[fges] d=20 deg=1.0 trial=6  shd=2  edges_true=4  edges_est=6  rt=2.878s
[fges] d=20 deg=1.0 trial=7  shd=3  edges_true=15  edges_est=18  rt=2.941s
[fges] d=20 deg=1.0 trial=8  shd=4  edges_true=5  edges_est=10  rt=3.041s
[fges] d=20 deg=1.0 trial=9  shd=7  edges_true=9  edges_est=16  rt=2.993s
[fges] d=20 deg=1.0 trial=10  shd=6  edges_true=10  edges_est=16  rt=2.921s
[fges] d=20 deg=1.0 trial=11  shd=7  edges_true=18  edges_est=25  rt=2.827s
[fges] d=20 deg=1.0 trial=12  shd=6  edges_true=6  edges_est=12  rt=2.593s
[fges] d=20 deg=1.0 trial=13  shd=6  edges_true=7  edges_est=12  rt=2.530s
[fges] d=20 deg=1.0 trial=14 

In [12]:
# 5) 汇总：SHD 均值 / 标准差 透视表
# ──────────────────────────────────────────────────────────────────────────────

if len(df_trials) == 0:
    print('没有可汇总的结果。')
else:
    df_summary = (
        df_trials
        .groupby(['d', 'degree'], as_index=False)
        .agg(
            shd_mean        = ('shd',         'mean'),
            shd_std         = ('shd',         'std'),
            shd_min         = ('shd',         'min'),
            shd_max         = ('shd',         'max'),
            n_edges_true    = ('n_edges_true','mean'),
            n_edges_est     = ('n_edges_est', 'mean'),
            runtime_sec     = ('runtime_sec', 'mean'),
            trials          = ('trial_id',    'count'),
        )
        .sort_values(['d', 'degree'])
        .reset_index(drop=True)
    )

    # 透视表：行=degree, 列=d, 值=SHD 均值
    pivot_mean = df_trials.pivot_table(
        index='degree', columns='d', values='shd', aggfunc='mean'
    ).round(2)
    pivot_std = df_trials.pivot_table(
        index='degree', columns='d', values='shd', aggfunc='std'
    ).round(2)

    # 合成 "mean ± std" 格式表
    pivot_combo = pivot_mean.copy().astype(str)
    for _i in pivot_mean.index:
        for _j in pivot_mean.columns:
            _m = pivot_mean.loc[_i, _j]
            _s = pivot_std.loc[_i, _j]
            pivot_combo.loc[_i, _j] = f'{_m:.2f} ± {_s:.2f}' if pd.notna(_m) else '—'

    print('===== SHD 均值（行=degree, 列=d）=====')
    display(pivot_mean)

    print('\n===== SHD 标准差（行=degree, 列=d）=====')
    display(pivot_std)

    print('\n===== SHD: mean ± std =====')
    display(pivot_combo)

    print('\n===== 详细汇总（每 (d, degree) 一行）=====')
    display(
        df_summary.style.format({
            'shd_mean':     '{:.2f}',
            'shd_std':      '{:.2f}',
            'shd_min':      '{:.0f}',
            'shd_max':      '{:.0f}',
            'n_edges_true': '{:.1f}',
            'n_edges_est':  '{:.1f}',
            'runtime_sec':  '{:.3f}',
        })
    )

===== SHD 均值（行=degree, 列=d）=====


d,20,30,40,50
degree,,,,
1.0,4.60,8.50,11.60,14.60
2.0,4.85,10.65,12.00,15.35
3.0,18.15,23.30,16.75,26.50
4.0,33.20,44.80,81.15,90.10



===== SHD 标准差（行=degree, 列=d）=====


d,20,30,40,50
degree,,,,
1.0,1.98,2.78,3.05,3.83
2.0,2.16,7.07,5.52,7.49
3.0,20.48,32.60,14.64,27.57
4.0,26.11,39.02,79.83,103.53



===== SHD: mean ± std =====


d,20,30,40,50
degree,,,,
1.0,4.60 ± 1.98,8.50 ± 2.78,11.60 ± 3.05,14.60 ± 3.83
2.0,4.85 ± 2.16,10.65 ± 7.07,12.00 ± 5.52,15.35 ± 7.49
3.0,18.15 ± 20.48,23.30 ± 32.60,16.75 ± 14.64,26.50 ± 27.57
4.0,33.20 ± 26.11,44.80 ± 39.02,81.15 ± 79.83,90.10 ± 103.53



===== 详细汇总（每 (d, degree) 一行）=====


,d,degree,shd_mean,shd_std,shd_min,shd_max,n_edges_true,n_edges_est,runtime_sec,trials
0,20,1.000000,4.60,1.98,1,7,9.8,14.4,2.795,20
1,20,2.000000,4.85,2.16,2,8,20.1,24.9,3.150,20
2,20,3.000000,18.15,20.48,2,71,28.6,43.7,3.737,20
3,20,4.000000,33.20,26.11,3,89,38.6,65.8,3.995,20
4,30,1.000000,8.50,2.78,4,14,16.8,25.0,5.450,20
5,30,2.000000,10.65,7.07,3,34,30.0,40.0,5.451,20
6,30,3.000000,23.30,32.60,2,107,44.4,64.3,5.583,20
7,30,4.000000,44.80,39.02,3,128,58.5,96.3,5.307,20
8,40,1.000000,11.60,3.05,6,17,18.8,30.8,6.370,20
9,40,2.000000,12.00,5.52,6,27,39.9,51.1,5.517,20


In [13]:
# 6) 保存结果
# ──────────────────────────────────────────────────────────────────────────────

if len(df_trials) > 0:
    _ts_str = datetime.now().strftime('%Y%m%d_%H%M%S')
    _trials_path  = os.path.join(CFG['out_dir'], f"{CFG['tag']}_trials_{_ts_str}.csv")
    _summary_path = os.path.join(CFG['out_dir'], f"{CFG['tag']}_summary_{_ts_str}.csv")
    _pivot_path   = os.path.join(CFG['out_dir'], f"{CFG['tag']}_pivot_{_ts_str}.csv")

    df_trials.to_csv(_trials_path,  index=False)
    df_summary.to_csv(_summary_path, index=False)
    pivot_mean.to_csv(_pivot_path)

    # latest
    df_trials.to_csv(os.path.join(CFG['out_dir'], f"{CFG['tag']}_trials.csv"),  index=False)
    df_summary.to_csv(os.path.join(CFG['out_dir'], f"{CFG['tag']}_summary.csv"), index=False)
    pivot_mean.to_csv(os.path.join(CFG['out_dir'], f"{CFG['tag']}_pivot.csv"))

    print(f'Trials  保存至: {_trials_path}')
    print(f'Summary 保存至: {_summary_path}')
    print(f'Pivot   保存至: {_pivot_path}')
else:
    print('无结果可保存。')

if _SKIP_LOGS:
    print(f'\n跳过记录 ({len(_SKIP_LOGS)} 条):')
    display(pd.DataFrame(_SKIP_LOGS))

Trials  保存至: c:\Users\super\DAG\experiments\results\er_fges_d_degree_trials_20260412_174329.csv
Summary 保存至: c:\Users\super\DAG\experiments\results\er_fges_d_degree_summary_20260412_174329.csv
Pivot   保存至: c:\Users\super\DAG\experiments\results\er_fges_d_degree_pivot_20260412_174329.csv
